# Task 5 — Exploratory Data Analysis (EDA): Titanic Dataset

**Objective:** Explore the Titanic passenger dataset using statistics and visualizations, identify patterns/trends/anomalies, and summarize findings.

**Tools:** Python, Pandas, Matplotlib, Seaborn.

> **Dataset note:** This notebook downloads the public Seaborn Titanic CSV. Internet access is required the first time you run it. If you already have `titanic.csv`, place it in this project folder and the notebook will use it.


In [ ]:
import os
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
LOCAL_FILE = "titanic.csv"

if os.path.exists(LOCAL_FILE):
    df = pd.read_csv(LOCAL_FILE)
else:
    try:
        df = pd.read_csv(DATA_URL)
        df.to_csv(LOCAL_FILE, index=False)
    except Exception as exc:
        raise RuntimeError(
            "Could not download the dataset. Download titanic.csv from the public "
            "Seaborn-data repository and place it next to this notebook, then rerun."
        ) from exc

print("Dataset loaded successfully.")
print("Rows, columns:", df.shape)
display(df.head())


## 1. Data overview and quality

In [ ]:
df.info()


In [ ]:
display(df.describe(include="all").T)


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_percent": missing_pct})
display(missing_summary[missing_summary["missing_count"] > 0])

print("Duplicate rows:", df.duplicated().sum())


**Observation:** `describe()` summarizes numeric and categorical fields, while `info()` shows data types and non-null counts. Missing values are especially important for `deck`, `age`, and `embarked` in this dataset. Missing values should be handled according to the analysis goal; they should not automatically be replaced with zero.

## 2. Univariate analysis

In [ ]:
# Survival counts and passenger class counts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="survived", ax=axes[0])
axes[0].set_title("Passenger survival (0 = no, 1 = yes)")
sns.countplot(data=df, x="pclass", ax=axes[1])
axes[1].set_title("Passengers by ticket class")
plt.tight_layout()
plt.show()

print("Survival counts:")
display(df["survived"].value_counts(dropna=False))
print("Sex counts:")
display(df["sex"].value_counts(dropna=False))
print("Class counts:")
display(df["pclass"].value_counts(dropna=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="age", bins=25, kde=True, ax=axes[0])
axes[0].set_title("Age distribution")
sns.boxplot(data=df, x="age", ax=axes[1])
axes[1].set_title("Age boxplot")
plt.tight_layout()
plt.show()


**Observations:** The survival count plot compares the number of survivors and non-survivors. The age histogram shows the shape of the age distribution; the boxplot helps reveal unusually low or high ages. Missing age values mean the age charts represent passengers with recorded ages only.

## 3. Bivariate analysis

In [ ]:
# Survival rate by sex and passenger class
survival_by_sex = df.groupby("sex", observed=False)["survived"].mean().sort_values(ascending=False)
survival_by_class = df.groupby("pclass", observed=False)["survived"].mean().sort_index()
print("Survival rate by sex:")
display((survival_by_sex * 100).round(1).rename("survival_percent"))
print("Survival rate by passenger class:")
display((survival_by_class * 100).round(1).rename("survival_percent"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x="sex", y="survived", errorbar=None, ax=axes[0])
axes[0].set_title("Survival rate by sex")
axes[0].set_ylabel("Mean survival (proportion)")
sns.barplot(data=df, x="pclass", y="survived", errorbar=None, ax=axes[1])
axes[1].set_title("Survival rate by passenger class")
axes[1].set_ylabel("Mean survival (proportion)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="survived", y="age")
plt.title("Age by survival status")
plt.xlabel("Survived (0 = no, 1 = yes)")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="fare", hue="survived", alpha=0.65)
plt.title("Age vs fare, coloured by survival")
plt.show()


**Observations:** Compare survival proportions rather than raw counts when groups have different sizes. In this dataset, women had a substantially higher survival rate than men, and first-class passengers generally had higher survival rates than passengers in lower classes. These are associations in the observed data and do not, by themselves, prove causation. The age-versus-fare scatterplot can show broad patterns but is affected by overlapping passengers and extreme fares.

## 4. Multivariate analysis and correlations

In [ ]:
numeric_df = df.select_dtypes(include="number")
corr = numeric_df.corr(numeric_only=True)
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation heatmap for numeric variables")
plt.show()


In [ ]:
# Pairplot on a manageable subset of numeric variables
pair_data = df[["survived", "pclass", "age", "fare", "sibsp", "parch"]].dropna()
sns.pairplot(pair_data, corner=True, diag_kind="hist")
plt.suptitle("Pairplot of selected numeric variables", y=1.02)
plt.show()


**Observations:** The heatmap summarizes pairwise linear correlations among numeric columns. A pairplot shows distributions on the diagonal and pairwise relationships off the diagonal. Ticket class and fare tend to be related because class 1 is a higher class than class 3 and often has higher fares. Because `survived` is binary and `pclass` is ordinal-coded, correlation values should be interpreted carefully. For formal multicollinearity checks in a predictive model, consider variance inflation factor (VIF) alongside domain knowledge.

## 5. Outliers and skewness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["fare"].dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title("Fare distribution")
sns.boxplot(x=df["fare"], ax=axes[1])
axes[1].set_title("Fare boxplot")
plt.tight_layout()
plt.show()

print("Fare skewness:", round(df["fare"].skew(), 3))
print("Age skewness:", round(df["age"].skew(), 3))


**Observation:** Fare data is typically right-skewed, with a small number of passengers paying much higher fares. Extreme values should be investigated rather than removed automatically. Depending on the analysis, options include a log transformation (`log1p`), robust statistics, or retaining the original values and using methods less sensitive to outliers.

## 6. Key findings (summary)

In [ ]:
survival_rate = df["survived"].mean() * 100
print(f"Overall survival rate: {survival_rate:.1f}%")
print("\nSurvival rate by sex (%):")
display((df.groupby("sex", observed=False)["survived"].mean() * 100).round(1))
print("\nSurvival rate by passenger class (%):")
display((df.groupby("pclass", observed=False)["survived"].mean() * 100).round(1))
print("\nMissing values (top 10):")
display(df.isna().sum().sort_values(ascending=False).head(10))


### Conclusion
- EDA combines summaries and visualizations to understand data quality, distributions, relationships, and unusual values before formal modelling.
- Survival was not evenly distributed across passenger groups; sex and ticket class showed clear associations with survival.
- Age and fare contain missing values or skewness that need deliberate handling.
- Correlation heatmaps and pairplots are useful exploratory tools, but association does not imply causation.
- Any imputation, transformation, or row removal should be documented and justified.

### Reproducibility
The notebook uses the public Seaborn Titanic dataset. Save the notebook and exported PDF report together with a README in your GitHub repository.